# Synthetic Product Growth Learning Lab

This notebook uses the Data Analytics plugin's bundled synthetic demo. It demonstrates a reproducible SQLite workflow over a CSV snapshot.


## tl;dr

- Weekly signups rose **38.0%** and revenue rose **31.8%** from the first to the last week.
- Overall activation fell **3.7 percentage points**.
- Paid Search activation fell **17.3 percentage points** while support tickets rose **160%**, making it the clearest issue to investigate.


## Context & Methods

### Key Assumptions

- Each row is one acquisition channel for one complete week.
- Activation and paid-conversion rates use signups as the denominator.
- First-to-last comparisons are descriptive; the synthetic data does not establish causality.


In [1]:
import csv, json, sqlite3
from pathlib import Path

data_path = Path('sample-product-growth.csv')
rows = list(csv.DictReader(data_path.open(newline='', encoding='utf-8')))
connection = sqlite3.connect(':memory:')
connection.execute('''CREATE TABLE product_growth (
  week_start TEXT, acquisition_channel TEXT, signups INTEGER,
  activated_users INTEGER, paid_conversions INTEGER,
  revenue_usd REAL, support_tickets INTEGER)''')
connection.executemany('INSERT INTO product_growth VALUES (?, ?, ?, ?, ?, ?, ?)', [
  (r['week_start'], r['acquisition_channel'], int(r['signups']),
   int(r['activated_users']), int(r['paid_conversions']),
   float(r['revenue_usd']), int(r['support_tickets'])) for r in rows
])
print(f'Loaded {len(rows)} synthetic rows from {data_path.name}')


Loaded 32 synthetic rows from sample-product-growth.csv


## Data

### 1. Aggregate weekly funnel performance


In [2]:
weekly_sql = 'SELECT\n  week_start,\n  SUM(signups) AS signups,\n  SUM(activated_users) AS activated_users,\n  ROUND(1.0 * SUM(activated_users) / SUM(signups), 4) AS activation_rate,\n  SUM(paid_conversions) AS paid_conversions,\n  ROUND(1.0 * SUM(paid_conversions) / SUM(signups), 4) AS signup_to_paid_rate,\n  SUM(revenue_usd) AS revenue_usd,\n  SUM(support_tickets) AS support_tickets,\n  ROUND(100.0 * SUM(support_tickets) / SUM(signups), 2) AS tickets_per_100_signups\nFROM product_growth\nGROUP BY week_start\nORDER BY week_start'
cursor = connection.execute(weekly_sql)
weekly_columns = [item[0] for item in cursor.description]
weekly = [dict(zip(weekly_columns, row)) for row in cursor.fetchall()]
print(json.dumps(weekly, indent=2))


[
  {
    "week_start": "2026-04-06",
    "signups": 1080,
    "activated_users": 661,
    "activation_rate": 0.612,
    "paid_conversions": 221,
    "signup_to_paid_rate": 0.2046,
    "revenue_usd": 45600.0,
    "support_tickets": 80,
    "tickets_per_100_signups": 7.41
  },
  {
    "week_start": "2026-04-13",
    "signups": 1122,
    "activated_users": 692,
    "activation_rate": 0.6168,
    "paid_conversions": 231,
    "signup_to_paid_rate": 0.2059,
    "revenue_usd": 47680.0,
    "support_tickets": 79,
    "tickets_per_100_signups": 7.04
  },
  {
    "week_start": "2026-04-20",
    "signups": 1169,
    "activated_users": 729,
    "activation_rate": 0.6236,
    "paid_conversions": 244,
    "signup_to_paid_rate": 0.2087,
    "revenue_usd": 50400.0,
    "support_tickets": 85,
    "tickets_per_100_signups": 7.27
  },
  {
    "week_start": "2026-04-27",
    "signups": 1219,
    "activated_users": 764,
    "activation_rate": 0.6267,
    "paid_conversions": 256,
    "signup_to_paid_rate":

## Results

### 2. Compare channel quality and growth


In [3]:
channel_sql = 'WITH channel_baselines AS (\n  SELECT\n    acquisition_channel,\n    MAX(CASE WHEN week_start = (SELECT MIN(week_start) FROM product_growth)\n      THEN signups END) AS first_signups,\n    MAX(CASE WHEN week_start = (SELECT MAX(week_start) FROM product_growth)\n      THEN signups END) AS last_signups,\n    1.0 * MAX(CASE WHEN week_start = (SELECT MIN(week_start) FROM product_growth)\n      THEN activated_users END)\n      / MAX(CASE WHEN week_start = (SELECT MIN(week_start) FROM product_growth)\n      THEN signups END) AS first_activation_rate,\n    1.0 * MAX(CASE WHEN week_start = (SELECT MAX(week_start) FROM product_growth)\n      THEN activated_users END)\n      / MAX(CASE WHEN week_start = (SELECT MAX(week_start) FROM product_growth)\n      THEN signups END) AS last_activation_rate,\n    1.0 * MAX(CASE WHEN week_start = (SELECT MIN(week_start) FROM product_growth)\n      THEN paid_conversions END)\n      / MAX(CASE WHEN week_start = (SELECT MIN(week_start) FROM product_growth)\n      THEN signups END) AS first_paid_rate,\n    1.0 * MAX(CASE WHEN week_start = (SELECT MAX(week_start) FROM product_growth)\n      THEN paid_conversions END)\n      / MAX(CASE WHEN week_start = (SELECT MAX(week_start) FROM product_growth)\n      THEN signups END) AS last_paid_rate,\n    MAX(CASE WHEN week_start = (SELECT MIN(week_start) FROM product_growth)\n      THEN revenue_usd END) AS first_revenue,\n    MAX(CASE WHEN week_start = (SELECT MAX(week_start) FROM product_growth)\n      THEN revenue_usd END) AS last_revenue,\n    MAX(CASE WHEN week_start = (SELECT MIN(week_start) FROM product_growth)\n      THEN support_tickets END) AS first_tickets,\n    MAX(CASE WHEN week_start = (SELECT MAX(week_start) FROM product_growth)\n      THEN support_tickets END) AS last_tickets\n  FROM product_growth\n  GROUP BY acquisition_channel\n)\nSELECT\n  acquisition_channel,\n  first_signups,\n  last_signups,\n  ROUND(1.0 * last_signups / first_signups - 1, 4) AS signup_growth,\n  ROUND(first_activation_rate, 4) AS first_activation_rate,\n  ROUND(last_activation_rate, 4) AS last_activation_rate,\n  ROUND(last_activation_rate - first_activation_rate, 4) AS activation_rate_change,\n  ROUND(100.0 * (last_activation_rate - first_activation_rate), 2) AS activation_rate_change_pp,\n  ROUND(first_paid_rate, 4) AS first_paid_rate,\n  ROUND(last_paid_rate, 4) AS last_paid_rate,\n  ROUND(last_paid_rate - first_paid_rate, 4) AS paid_rate_change,\n  ROUND(100.0 * (last_paid_rate - first_paid_rate), 2) AS paid_rate_change_pp,\n  first_revenue,\n  last_revenue,\n  ROUND(1.0 * last_revenue / first_revenue - 1, 4) AS revenue_growth,\n  first_tickets,\n  last_tickets,\n  ROUND(1.0 * last_tickets / first_tickets - 1, 4) AS ticket_growth\nFROM channel_baselines\nORDER BY revenue_growth DESC'
cursor = connection.execute(channel_sql)
channel_columns = [item[0] for item in cursor.description]
channels = [dict(zip(channel_columns, row)) for row in cursor.fetchall()]
print(json.dumps(channels, indent=2))


[
  {
    "acquisition_channel": "Partners",
    "first_signups": 120,
    "last_signups": 205,
    "signup_growth": 0.7083,
    "first_activation_rate": 0.6833,
    "last_activation_rate": 0.7366,
    "activation_rate_change": 0.0533,
    "activation_rate_change_pp": 5.33,
    "first_paid_rate": 0.2917,
    "last_paid_rate": 0.3024,
    "paid_rate_change": 0.0108,
    "paid_rate_change_pp": 1.08,
    "first_revenue": 8400.0,
    "last_revenue": 14880.0,
    "revenue_growth": 0.7714,
    "first_tickets": 10,
    "last_tickets": 14,
    "ticket_growth": 0.4
  },
  {
    "acquisition_channel": "Referral",
    "first_signups": 180,
    "last_signups": 245,
    "signup_growth": 0.3611,
    "first_activation_rate": 0.7,
    "last_activation_rate": 0.751,
    "activation_rate_change": 0.051,
    "activation_rate_change_pp": 5.1,
    "first_paid_rate": 0.2667,
    "last_paid_rate": 0.3061,
    "paid_rate_change": 0.0395,
    "paid_rate_change_pp": 3.95,
    "first_revenue": 9600.0,
    "last_

## Takeaways

1. Preserve growth, but do not evaluate acquisition on signup volume alone.
2. Investigate Paid Search targeting, landing pages, and onboarding because its volume rose while downstream quality deteriorated.
3. Test incremental investment in Partners and Referral, where signup growth coincided with stronger activation and revenue.
